In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords
from datetime import datetime

In [ ]:
# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
# Load dataset
file_path = "/content/reddit_posts_latest_08_February_2025.csv"
df = pd.read_csv(file_path)

In [ ]:
# 1. Handle Missing Values
df.dropna(subset=['Author', 'Selftext', 'Title', 'Subreddit'], inplace=True)
df.fillna("", inplace=True)  # Fill other missing values with empty string

In [ ]:
# 2. Convert Data Types
df['Created_UTC'] = df['Created_UTC'].astype(int)  # Ensure UNIX timestamp format
df['Created_DateTime'] = pd.to_datetime(df['Created_DateTime'])  # Convert to datetime
df['Score'] = df['Score'].astype(int)  # Ensure Score is integer

In [ ]:
# 3. Remove Duplicates
df.drop_duplicates(inplace=True)

In [ ]:
# 4. Normalize Text Data
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['Title'] = df['Title'].apply(clean_text)
df['Selftext'] = df['Selftext'].apply(clean_text)

In [ ]:
# Ensure Tokenized_Selftext is properly formatted as a list
def format_tokenized_text(text):
    try:
        tokens = ast.literal_eval(text) if isinstance(text, str) else text
        if isinstance(tokens, list):
            return [t.lower() for t in tokens if isinstance(t, str)]
        return []
    except:
        return []

df['Tokenized_Selftext'] = df['Tokenized_Selftext'].apply(format_tokenized_text)

In [ ]:
# 5. Handle Outliers in Score
q_low = df['Score'].quantile(0.01)
q_high = df['Score'].quantile(0.99)
df = df[(df['Score'] >= q_low) & (df['Score'] <= q_high)]

In [ ]:
# 6. Standardize Subreddit Column
df['Subreddit'] = df['Subreddit'].str.strip().str.lower()

In [ ]:
# 7. Remove Stopwords from Tokenized_Selftext
df['Tokenized_Selftext'] = df['Tokenized_Selftext'].apply(lambda tokens: [t for t in tokens if t not in stop_words])

In [ ]:
# 8. Save Cleaned Data
cleaned_file_path = "/content/cleaned_reddit_posts_latest_08_February_2025.csv"
df.to_csv(cleaned_file_path, index=False)

In [ ]:
print(f"Cleaned data saved to {cleaned_file_path}")